In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 5)

In [ ]:
def train_poly_models(X, y, max_degree=8, learning_rate=0.01, num_iterations=100):
    """Train polynomial models of degrees 1 through max_degree, compare"""

    # Normalise X
    X_mean, X_std = X.mean(), X.std()
    X_norm = (X - X_mean) / X_std

    # Pre-calculate features for all degrees to slice later
    X_poly_full = np.array([[x**i for i in range(max_degree, -1, -1)] for x in X_norm])
    # Shape: (num_samples, max_degree+1)

    # Storage for train_results
    train_results = {
        'final_weights': [],
        'degrees': [],
        'train_loss': []
    }

    test_results = {
        'degrees': [],
        'test_loss': []
    }

    print("X_norm = ", X_norm)
    print("y = ", y)
    np.set_printoptions(precision=3)
    print("X_poly_full shape ", X_poly_full.shape)
    
    # Train models of each degree, slicing X_poly_full as needed
    for degree in range(1, max_degree+1):
        print(f"Training degree {degree} model...")

        # Only extract features needed for this model's degree
        # For degree d, we need last (d+1) columns: [X^d, X^(d-1), ..., X^0]
        X_poly = X_poly_full[:, -(degree+1):]  # Shape: (num_samples, degree+1)

        # Initialise weights for this degree
        weights = np.zeros(degree + 1) # degree+1 to include degree 0

        # Training!
        for i in range(num_iterations):
            y_pred = X_poly @ weights

            # Loss
            loss = np.mean((y - y_pred) ** 2)
            gradients = -2 * X_poly.T @ (y - y_pred) / num_samples
            weights -= learning_rate * gradients

        # Store train_results after each model training
        train_results['final_weights'].append(weights.copy())
        train_results['degrees'].append(degree)
        train_results['train_loss'].append(loss)

    print("final degrees: ", train_results['degrees'])
    
    return train_results, X_mean, X_std

def test_poly_models(X, y, max_degree, final_weights):
    """Test polynomial models of degrees 1 through max_degree"""

    # Normalise X
    X_mean, X_std = X.mean(), X.std()
    X_norm = (X - X_mean) / X_std

    # Pre-calculate features for all degrees to slice later
    X_poly_full = np.array([[x**i for i in range(max_degree, -1, -1)] for x in X_norm])
    # Shape: (num_samples, max_degree+1)
    
    # Calculate test loss for each model given weights
    for degree in range(1, max_degree+1):
        print(f"Calculating test loss for degree {degree} model...")

        # Only extract features needed for this model's degree
        # For degree d, we need last (d+1) columns: [X^d, X^(d-1), ..., X^0]
        X_poly = X_poly_full[:, -(degree+1):]  # Shape: (num_samples, degree+1)

        # Testing! @LIYA FLAG IS THIS CORRECT?
        y_pred = X_poly @ final_weights[degree]

        # Loss
        loss = np.mean((y - y_pred) ** 2)
        
        # Store train_results after each model training
        test_results['degrees'].append(degree)
        test_results['test_loss'].append(loss)

    print("final degrees: ", test_results['degrees'])
    
    return test_results, X_mean, X_std

def plot_degree_comparison(X, y, train_results, X_mean, X_std, max_degree):
    n_models = len(train_results['final_weights'])

    # Create subplots (e.g., 3 rows × 5 columns for 15 models)
    n_cols = 5
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    axes = axes.flatten()

    # Many X-points for smooth curve plotting
    X_plot = np.linspace(X.min(), X.max(), 200)
    X_plot_norm = (X_plot - X_mean) / X_std  # Normalize for plotting
    X_poly_plot_full = np.array([[x**i for i in range(n_models, -1, -1)] for x in X_plot_norm])

    for idx in range(n_models):
        degree = len(train_results['final_weights'][idx]) - 1 # for 1 weights vector, num of weights --> degree + 1! (since X^0)
        weights = train_results['final_weights'][idx]
        
        ax = axes[idx]

        # Plot training data
        ax.scatter(X, y, alpha=0.7, s=100, label='Data', color='blue', zorder=3)
        
        # Plot fitted curve
        X_plot_poly = X_poly_plot_full[:, -(degree+1):]  # Extract features for this degree
        y_plot = X_plot_poly @ weights

        ax.plot(X_plot, y_plot, 'r-', linewidth=2.5, label=f'Fit', zorder=1)
        
        # Formatting
        ax.set_title(f'Degree {degree}\nLoss: {train_results["train_loss"][idx]:.2f}', 
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('y')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        ax.set_ylim([y.min() - 20, y.max() + 20])
        # ax.set_ylim([-100, 10])  
        
        # # Highlight overfitting cases (degree >= n_samples)
        # if degree >= len(X):
        #     ax.set_facecolor('#fff5f5')  # Light red background
        #     ax.set_title(f'Degree {degree} (OVERFIT!)\nLoss: {train_results["train_loss"][idx]:.2f}', 
        #                 fontsize=11, fontweight='bold', color='red')

    # Hide unused subplots
    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')

    fig.suptitle(f'Multinom Regression, Degrees 1 to {max_degree} over {len(X)} samples (gen from true degree 5)', 
             fontsize=24, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    # Plot final loss vs degree
    np.set_printoptions(precision=3)
    loss_min = min(train_results['train_loss'])
    loss_max = max(train_results['train_loss'])
    # print("train_loss array:", train_results['train_loss'])
    print("min loss: ", loss_min)
    print("max loss: ", loss_max)
    # print("Number of losses:", len(train_results['train_loss']))
    # print("Number of weight sets:", len(train_results['final_weights']))

    ax.plot(train_results['degrees'], train_results['train_loss'], 'o-', linewidth=2.5, markersize=8)
    # ax.axvline(x=len(X), color='red', linestyle='--', linewidth=2, 
    #             label=f'n_samples={len(X)}')
    ax.set_xlabel('Degree of model fitted', fontsize=12)
    ax.set_ylabel('Final loss (MSE)', fontsize=12)
    ax.set_yscale('log')
    ax.set_ylim([loss_min - 20, loss_max + 20])
    
    ax.set_title('Training loss vs. Degree of model fitted', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()        
 

In [ ]:
# Hyperparameters
max_degree = 12
noisiness = 2
learning_rate = 0.001
num_iterations = 200
num_samples = 20
test_split = 0.3

# Base data
np.random.seed(42)
X_train = np.linspace(0, 10, int(num_samples*(1-test_ratio))) # X training data

true_rel = -0.001*X**5 + 0.002*X**4 - 0.08*X**3 + 0.7*X**2 - 0.4*X + 1
y = true_rel + np.random.randn(num_samples) * noisiness

# Train models
train_results, X_mean, X_std = train_poly_models(X, y, max_degree, learning_rate, num_iterations)

# Visualise
plot_degree_comparison(X, y, train_results, X_mean, X_std, max_degree)